In [22]:
!pip install gymnasium stable-baselines3 pandas numpy


Defaulting to user installation because normal site-packages is not writeable


In [23]:
import os
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_checker import check_env
from npc_environment import NPCBehaviorEnv, ACTION_MAP, calculate_reward

In [3]:
os.makedirs(r'C:\Users\lanaa\Downloads\models', exist_ok=True)
train_df = pd.read_csv(r"C:\Users\lanaa\Downloads\train.csv")
print('Training data shape:', train_df.shape)
print('Training episodes:', train_df['episode_id'].nunique())

Training data shape: (69665, 25)
Training episodes: 1792


In [25]:
env = NPCBehaviorEnv(train_df)
check_env(env, warn=True)
obs, info = env.reset(seed=42)

print('Observation shape:', obs.shape)

print('\nAction reward sanity check:')
for action_id, action_name in ACTION_MAP.items():
    test_env = NPCBehaviorEnv(train_df)
    test_env.reset(seed=42)
    _, reward, _, _, _ = test_env.step(action_id)
    print(f'{action_id} - {action_name:12s}: {reward:7.2f}')

Observation shape: (12,)

Action reward sanity check:
0 - ATTACK      :   -2.00
1 - CHASE       :    0.00
2 - PATROL      :    0.00
3 - RETREAT     :    0.00
4 - SEARCH      :    2.00
5 - TAKE_COVER  :    0.00


In [26]:
train_env = DummyVecEnv([lambda: NPCBehaviorEnv(train_df)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

In [10]:
model = PPO(
    'MlpPolicy',
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02,
    verbose=1,
    seed=42
)
print('PPO model created.')

Using cpu device
PPO model created.


In [11]:
TOTAL_TIMESTEPS = 100_000
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print('PPO training completed.')

Starting PPO training for 100,000 timesteps...
-----------------------------
| time/              |      |
|    fps             | 687  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 581         |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013206972 |
|    clip_fraction        | 0.145       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.78       |
|    explained_variance   | -0.222      |
|    learning_rate        | 0.0003      |
|    loss                 | 0.124       |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0224     |
|    value_loss           | 0.596       |
-----------------------------

In [12]:
model.save(r'C:\Users\lanaa\Downloads\models\ppo_npc_simulator')
train_env.save(r'C:\Users\lanaa\Downloads\models/vec_normalize_simulator.pkl')
print('PPO model saved.')
print('VecNormalize statistics saved.')

PPO model saved.
VecNormalize statistics saved.


In [28]:
validation_df = pd.read_csv(r"C:\Users\lanaa\Downloads\validation.csv")
eval_env_raw = DummyVecEnv([lambda: NPCBehaviorEnv(validation_df)])
eval_env = VecNormalize.load(r"C:\Users\lanaa\Downloads\models\vec_normalize_simulator.pkl", eval_env_raw)

eval_env.training = False
eval_env.norm_reward = False

model = PPO.load(r"C:\Users\lanaa\Downloads\models\ppo_npc_simulator", env=eval_env)
print('Validation environment and PPO model loaded.')

Validation environment and PPO model loaded.


In [30]:
NUM_EPISODES = 50
episode_rewards = []
episode_lengths = []
action_counts = {name: 0 for name in ACTION_MAP.values()}

for episode in range(NUM_EPISODES):
    obs = eval_env.reset()
    done = False
    total_reward = 0.0
    steps = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action_id = int(action[0])
        action_counts[ACTION_MAP[action_id]] += 1
        obs, reward, done_array, info = eval_env.step(action)
        total_reward += float(reward[0])
        steps += 1
        done = bool(done_array[0])
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

print('---PPO SIMULATOR EVALUATION RESULTS---')

## Summarizing

print(f'Mean episode reward: {np.mean(episode_rewards):.2f}')
print(f'Std episode reward:  {np.std(episode_rewards):.2f}')
print(f'Mean episode length: {np.mean(episode_lengths):.2f}')
print(f'Min episode length:  {np.min(episode_lengths)}')
print(f'Max episode length:  {np.max(episode_lengths)}')

print('---ACTION DISTRIBUTION---')
total_actions = sum(action_counts.values())
for name, count in action_counts.items():
    pct = count / total_actions * 100 if total_actions else 0
    print(f'{name:12s}: {count:5d} ({pct:6.2f}%)')
print('\nTotal actions:', total_actions)

---PPO SIMULATOR EVALUATION RESULTS---
Mean episode reward: 96.26
Std episode reward:  38.98
Mean episode length: 43.10
Min episode length:  19
Max episode length:  60
---ACTION DISTRIBUTION---
ATTACK      :   328 ( 15.22%)
CHASE       :    44 (  2.04%)
PATROL      :    84 (  3.90%)
RETREAT     :   519 ( 24.08%)
SEARCH      :  1032 ( 47.89%)
TAKE_COVER  :   148 (  6.87%)

Total actions: 2155


In [31]:
results = {
    'mean_episode_reward': np.mean(episode_rewards),
    'std_episode_reward': np.std(episode_rewards),
    'mean_episode_length': np.mean(episode_lengths),
    'min_episode_length': np.min(episode_lengths),
    'max_episode_length': np.max(episode_lengths)
}
for name, count in action_counts.items():
    results[f'{name.lower()}_percentage'] = count / total_actions * 100 if total_actions else 0
pd.DataFrame([results]).to_csv(r"C:\Users\lanaa\Downloads\models\ppo_simulator_evaluation_results.csv", index=False)
print('Saved: ../models/ppo_simulator_evaluation_results.csv')

Saved: ../models/ppo_simulator_evaluation_results.csv


In [32]:
print("---SAMPLE PPO NPC BEHAVIOR---")

obs, info = test_env.reset()

for step in range(30):

    # PPO prediction
    action, _ = model.predict(obs, deterministic=True)
    action_id = int(action)

    # Action name
    action_name = ACTION_MAP[action_id]

    # Environment step
    obs, reward, terminated, truncated, info = test_env.step(action_id)

    # Observation values
    npc_health = obs[0] * 100
    npc_stamina = obs[1] * 100
    player_health = obs[2] * 100
    distance = obs[3] * 100

    print(
        f"Step {step+1:02d} | "
        f"Action: {action_name:12s} | "
        f"Reward: {float(reward):6.2f} | "
        f"NPC HP: {npc_health:6.1f} | "
        f"Player HP: {player_health:6.1f} | "
        f"Distance: {distance:6.1f}"
    )

    if terminated or truncated:
        print("\nEpisode ended.")
        break

print("=" * 60)

---SAMPLE PPO NPC BEHAVIOR---
Step 01 | Action: PATROL       | Reward:   0.00 | NPC HP: 4783.0 | Player HP: 7450.0 | Distance: 2198.0
Step 02 | Action: PATROL       | Reward:   0.00 | NPC HP: 4783.0 | Player HP: 7450.0 | Distance: 1841.0
Step 03 | Action: ATTACK       | Reward:   2.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1527.0
Step 04 | Action: PATROL       | Reward:  -3.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1273.0
Step 05 | Action: PATROL       | Reward:   0.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1441.0
Step 06 | Action: PATROL       | Reward:   0.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1441.0
Step 07 | Action: PATROL       | Reward:   0.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1638.0
Step 08 | Action: PATROL       | Reward:   0.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1832.0
Step 09 | Action: PATROL       | Reward:   0.00 | NPC HP: 4063.0 | Player HP: 7450.0 | Distance: 1832.0
Step 10 | Action: PATROL       | R